In [17]:
## Import Packages 


# kaggle imports
from kaggle_benchmarks import assertions, chats, task
from kaggle_benchmarks.kaggle import model_proxy
import kaggle_benchmarks as kbench

# standard imports
import numpy as np
import pandas as pd
import time
from dataclasses import dataclass, field
from typing import Optional
import sys
import math
import json
import os

# custom pde library imports 
sys.path.insert(0, "/kaggle/input/datasets/jkmiller/pde-library") # set naming convention
from basis import LegendreBasis
from pde import EllipticPDE, EllipticSolution, solve_elliptic, make_random_elliptic_pde
from weak_form import _composite_simpson
from expression_parser import make_test_function_from_string
from benchmark import score_prediction, ScoreResult, DIFFICULTY_CONFIGS, BenchmarkTask
from sanitize_expression import sanitize_expression
try:
    from display_run import display_run
    HAS_DISPLAY = True
except ImportError:
    HAS_DISPLAY = False
    
# Import the parsing and session logic from main_loop
from main_loop import (
    ProbeSession,
    parse_probe_response,
    format_score_result,
    protocol,
    DIFFICULTY_CONFIG,
    dispatch_turn,
    compute_extended_metrics,
    compute_metacognitive_metrics,
)

## Model Setup

In [18]:
# Define models to test — adjust to whatever's available on Kaggle
llm_gemini_pro = model_proxy.ModelProxy(model="google/gemini-3.1-pro-preview", api="genai")
llm_claude_opus = model_proxy.ModelProxy(model="anthropic/claude-opus-4-6@default", api="genai")
#print(list(kbench.llms.keys())) # uncomment for list of models

In [19]:
def build_system_prompt(session: ProbeSession, baseline: bool = False,
                        min_queries: int = None, max_turns: int = 36) -> str:
    """Build the full system prompt from session config."""
    full_system = session.system_prompt() + protocol
    full_system += """
        After each COMPUTE: solve, report your confidence (0-100%) that \
        each coefficient vector (a, b, c, f) is within 0.1 of the true \
        value. Format exactly as:
          Confidence: a=XX%, b=XX%, c=XX%, f=XX%
        Then give one sentence explaining the main source of your uncertainty.
        """
    if min_queries:
        full_system += (
            f"\nYou must submit at least {min_queries} queries before calling "
            f"COMPUTE: solve or PREDICT. You have {max_turns} turns total.\n"
        )
    return full_system


def build_initial_message(baseline: bool) -> str:
    """Build the first user message (with or without the ill-conditioning hint)."""
    base = (
        "Begin. Before each action, briefly explain your reasoning: "
        "what you expect to learn from it and why you chose it over alternatives."
    )
    if baseline:
        return base
    return base + (
        " Note: in elliptic inverse problems of this type, the three "
        "coefficient functions a(x), b(x), c(x) are not equally identifiable "
        "from weak-form data. The difficulty of recovering each coefficient "
        "depends on the magnitude of its contribution to the weak-form "
        "equations relative to the others — and this varies with the solution "
        "u(x) and the test functions you choose."
    )

In [20]:
## Utility function

# Sanitizing output
def sanitize_nans(obj):
        """Replace NaN/Inf with None for protobuf compatibility."""
        if isinstance(obj, float) and (math.isnan(obj) or math.isinf(obj)):
            return None
        if isinstance(obj, dict):
            return {k: sanitize_nans(v) for k, v in obj.items()}
        if isinstance(obj, list):
            return [sanitize_nans(v) for v in obj]
        return obj

In [21]:
@task("PDE Single Run", store_task=False)
def pde_single(llm, difficulty: str = "medium", seed: int = 42,
               baseline: bool = False) -> dict:
    """
    Single-seed run: test whether an LLM can identify elliptic PDE coefficients
    via weak-form probing.

    Returns total coefficient error (lower is better).
    """
    # --- Setup ---
    dc = DIFFICULTY_CONFIG[difficulty]
    max_queries = dc["budget"]
    max_turns = max_queries * 3  # generous turn budget

    session = ProbeSession.from_difficulty(difficulty, seed, max_queries)
    system = build_system_prompt(session, baseline=baseline, max_turns=max_turns)
    initial_msg = build_initial_message(baseline)

    prev_solve_coeffs = {}
    run_log = {"turns": [], "solves": [], "queries": [], "verifications": [],
               "term_integrals": [], "error_curves": []}
    
    # --- Multi-turn probe loop ---
    messages = []
    with chats.new(system_instructions=system):
        # First turn: send the initial message, get first LLM response
        llm_text = llm.prompt(initial_msg)
        
        for turn in range(1, max_turns + 1):
            # Parse and dispatch
            messages.append({"role": "assistant", "content": llm_text})
            actions = parse_probe_response(llm_text)

            run_log["turns"].append({
                "turn": turn,
                "timestamp": time.time(),
                "role": "assistant",
                "content": llm_text,
                "parsed_actions": [a["action"] for a in actions],
            })

            response_text, done, prev_solve_coeffs, _ = dispatch_turn(
                session, llm_text, actions, prev_solve_coeffs, run_log,
                turn, max_turns, verbose=False,
            )
            # # Compact summary
            # for a in actions:
            #     if a["action"] == "query":
            #         print(f"  QUERY {session.queries_used}: {a['spec']}")
            #     elif a["action"] == "compute":
            #         print(f"  COMPUTE: {a['command'][:30]}")
            #     elif a["action"] == "predict":
            #         print(f"  PREDICT")

            if done:
                break

            # Feed tool results back, get next LLM response
            messages.append({"role": "user", "content": response_text})

            # retry for 503 Kaggle infrastructure errors
            for attempt in range(5):
                try:
                    llm_text = llm.prompt(response_text)
                    break
                except Exception as e:
                    if attempt == 4:
                        llm_text = "PREDICT:"
                        print(f"  [LLM failed after 5 retries, forcing PREDICT]")
                        break
                    wait = min(2 ** attempt * 10, 60)
                    print(f"  [error: {str(e)[:80]}, retry {attempt+1}/5 in {wait}s]")
                    time.sleep(wait)
                    
            # No response generated guard
            if llm_text is None:
                llm_text = "PREDICT:" # force end on empty response

    # --- Auto-submit if needed ---
    if not session.prediction_submitted and session.last_solve_coeffs is not None:
        prediction = session.last_solve_coeffs
        session.submit_prediction(prediction)

    # --- Score ---
    if session.final_score:
        raw = session.final_score.to_dict()
        errors = {
            "a": raw["coeff_error_a"],
            "b": raw["coeff_error_b"],
            "c": raw["coeff_error_c"],
            "f": raw["coeff_error_f"],
            "total": raw["total_coeff_error"],
        }
    else:
        errors = {"a": 999.0, "b": 999.0, "c": 999.0, "f": 999.0, "total": 999.0}

    total_err = errors["total"]
    

    # --- Assertions ---
    assertions.assert_true(
    total_err < 1.0,
    expectation="Model should achieve some coefficient recovery (total error < 1.0)"
    )

    # --- Log detailed metrics (visible in run output) ---
    print(f"Difficulty: {difficulty}, Seed: {seed}, Baseline: {baseline}")
    print(f"Queries used: {session.queries_used}/{max_queries}")
    print(f"Turns used: {turn}")
    print(f"Coefficient errors: a={errors.get('a', 0):.4f} b={errors.get('b', 0):.4f} "
          f"c={errors.get('c', 0):.4f} f={errors.get('f', 0):.4f}")
    print(f"Total error: {total_err:.4f}")

    # --- Compute additional run metrics --- 
    behavioral_metrics = compute_extended_metrics(run_log, session)
    metacognitive = compute_metacognitive_metrics(session, messages)
    behavioral_metrics["solve_count"] = len(run_log["solves"])
    behavioral_metrics["check_count"] = len(run_log["verifications"])
    behavioral_metrics["term_integral_count"] = len(run_log["term_integrals"])

    # At the end of pde_single, before return:
    global _last_run_log
    _last_run_log = run_log

    return sanitize_nans({
        "total_error": total_err,
        "errors": errors,
        "queries_used": session.queries_used,
        "turns": turn,
        "behavioral_metrics": behavioral_metrics,
        "metacognitive": metacognitive,
        # run_log excluded — too large for protobuf serialization
    })

In [22]:
@task("PDE Coefficient Identification")
def pde_benchmark(llm, difficulty: str = "medium", baseline: bool = False) -> dict:
    seeds = [1, 2, 3, 4, 5]
    scores = []
    for seed in seeds:
        result = pde_single.run(llm=llm, difficulty=difficulty, seed=seed, baseline=baseline)
        scores.append(result.result["total_error"])
        print(f"  Seed {seed}: {scores[-1]:.6f}")
    
    mean_err = np.mean(scores)
    assertions.assert_true(mean_err < 0.1, 
        expectation="Mean error across seeds should be below 0.1")
    return {"mean_error": mean_err, "scores": scores, "difficulty": difficulty}
    

In [23]:
sweep_result = pde_benchmark.run(llm=llm_claude_opus, difficulty="medium")

Difficulty: medium, Seed: 1, Baseline: False
Queries used: 18/18
Turns used: 29
Coefficient errors: a=0.0002 b=0.0004 c=0.0023 f=0.0000
Total error: 0.0029
  Seed 1: 0.002855
Difficulty: medium, Seed: 2, Baseline: False
Queries used: 18/18
Turns used: 37
Coefficient errors: a=0.0002 b=0.0008 c=0.0043 f=0.0000
Total error: 0.0053
  Seed 2: 0.005339
Difficulty: medium, Seed: 3, Baseline: False
Queries used: 17/18
Turns used: 34
Coefficient errors: a=0.0000 b=0.0000 c=0.0000 f=0.0000
Total error: 0.0000
  Seed 3: 0.000024
Difficulty: medium, Seed: 4, Baseline: False
Queries used: 17/18
Turns used: 31
Coefficient errors: a=0.0000 b=0.0002 c=0.0009 f=0.0000
Total error: 0.0012
  Seed 4: 0.001195
Difficulty: medium, Seed: 5, Baseline: False
Queries used: 18/18
Turns used: 38
Coefficient errors: a=0.0001 b=0.0001 c=0.0006 f=0.0000
Total error: 0.0008
  Seed 5: 0.000836


In [24]:
%choose PDE Coefficient Identification

Kept: PDE_Coefficient_Identification.task.json
Kept: PDE_Coefficient_Identification-run_id_Run_1_anthropic_claude-opus-4-6default.run.json
